# Robust Time Series Anomaly Detection and Handling

This notebook implements advanced anomaly detection methods for time series data to improve forecast robustness. 

**Methods Covered:**
- **Structural Break Detection:** Chow Test and CUSUM Analysis.
- **Point Anomaly Detection:** Isolation Forest and DBSCAN.
- **Contextual Anomaly Detection:** Seasonal Decomposition.
- **Anomaly Handling:** Removal, Imputation, and Separate Modeling.
- **Visualization:** Components to explain anomalies to non-technical stakeholders.

## 1. Setup and Dependencies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.api import SimpleExpSmoothing
from statsmodels.formula.api import ols

# Settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

## 2. Data Generation with Anomalies

We'll generate a synthetic dataset with a clear seasonal pattern and inject various types of anomalies.

In [ ]:
def generate_anomalous_data(start_date='2022-01-01', periods=365*2):
    dates = pd.date_range(start=start_date, periods=periods, freq='D')
    data = pd.DataFrame({'date': dates})
    
    # Base seasonal data
    data['value'] = 100 + 20 * np.sin(2 * np.pi * data.index / 7) + 10 * np.cos(2 * np.pi * data.index / 30.5) + np.random.normal(0, 5, periods)
    
    # Inject anomalies
    # Point anomalies
    data.loc[100, 'value'] *= 2.5 # Spike
    data.loc[200, 'value'] /= 2.0 # Dip
    
    # Contextual anomaly (low sales on a typically high-sales day)
    data.loc[data.index % 7 == 6, 'value'] *= 1.5 # Boost weekends
    data.loc[146, 'value'] = 50 # A Saturday with low sales
    
    # Structural break
    break_point = 400
    data.loc[break_point:, 'value'] += 50
    
    return data.set_index('date')

df = generate_anomalous_data()

plt.figure(figsize=(18, 6))
plt.plot(df.index, df['value'])
plt.title('Synthetic Time Series with Anomalies')
plt.show()

## 3. Anomaly Detection Pipeline

### 3.1. Structural Break Detection (Chow Test)

In [ ]:
def chow_test(df, break_point):
    df['time'] = np.arange(len(df.index))
    df['indicator'] = (df['time'] > break_point).astype(int)
    df['time_indicator'] = df['time'] * df['indicator']
    
    # Unrestricted model (with break)
    model_unrestricted = ols('value ~ time + indicator + time_indicator', data=df).fit()
    
    # Restricted model (no break)
    model_restricted = ols('value ~ time', data=df).fit()
    
    # Chow test statistic
    k = 2 # Number of restrictions
    N = len(df)
    F_statistic = ((model_restricted.ssr - model_unrestricted.ssr) / k) / (model_unrestricted.ssr / (N - 2*k))
    
    return F_statistic, model_unrestricted.pvalues['indicator']

break_point_to_test = 400
f_stat, p_value = chow_test(df.copy(), break_point_to_test)
print(f"Chow Test at point {break_point_to_test}:")
print(f"  F-statistic: {f_stat:.2f}")
print(f"  P-value: {p_value:.4f}")
if p_value < 0.05:
    print("  Result: Significant structural break detected.")
else:
    print("  Result: No significant structural break detected.")

### 3.2. Point Anomaly Detection (Isolation Forest)

In [ ]:
def detect_isolation_forest(df):
    model = IsolationForest(contamination=0.01, random_state=42)
    df['anomaly_if'] = model.fit_predict(df[['value']])
    return df

df = detect_isolation_forest(df.copy())
anomalies_if = df[df['anomaly_if'] == -1]

plt.figure(figsize=(18, 6))
plt.plot(df.index, df['value'], label='Time Series')
plt.scatter(anomalies_if.index, anomalies_if['value'], color='red', label='Isolation Forest Anomalies')
plt.title('Point Anomaly Detection with Isolation Forest')
plt.legend()
plt.show()

### 3.3. Contextual Anomaly Detection (Seasonal Decomposition)

In [ ]:
def detect_seasonal_decomposition(df, period=7, threshold=2.0):
    decomposition = seasonal_decompose(df['value'], model='additive', period=period)
    df['residual'] = decomposition.resid
    df['anomaly_seasonal'] = (np.abs(df['residual']) > threshold * df['residual'].std()).astype(int)
    return df, decomposition

df, decomposition = detect_seasonal_decomposition(df.copy())
anomalies_seasonal = df[df['anomaly_seasonal'] == 1]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 10), sharex=True)
ax1.plot(df.index, df['value'], label='Time Series')
ax1.scatter(anomalies_seasonal.index, anomalies_seasonal['value'], color='red', label='Contextual Anomalies')
ax1.set_title('Contextual Anomaly Detection with Seasonal Decomposition')
ax1.legend()

ax2.plot(df.index, df['residual'], label='Residuals')
ax2.axhline(y=2*df['residual'].std(), color='r', linestyle='--', label='Threshold')
ax2.axhline(y=-2*df['residual'].std(), color='r', linestyle='--')
ax2.set_title('Residuals from Decomposition')
ax2.legend()
plt.show()

## 4. Anomaly Handling and Impact on Forecasting

In [ ]:
df_clean = df.copy()
df_clean.loc[df_clean['anomaly_if'] == -1, 'value'] = np.nan
df_clean.loc[df_clean['anomaly_seasonal'] == 1, 'value'] = np.nan
df_clean['value'] = df_clean['value'].interpolate(method='time')

from sklearn.metrics import mean_squared_error

def forecast_and_evaluate(df, title):
    train_size = int(len(df) * 0.8)
    train, test = df[0:train_size], df[train_size:len(df)]
    
    model = SimpleExpSmoothing(train['value'], initialization_method="estimated").fit()
    forecast = model.forecast(len(test))
    
    rmse = np.sqrt(mean_squared_error(test['value'], forecast))
    print(f"RMSE for {title}: {rmse:.2f}")
    
    plt.figure(figsize=(18, 6))
    plt.plot(train.index, train['value'], label='Train')
    plt.plot(test.index, test['value'], label='Test')
    plt.plot(test.index, forecast, label='Forecast')
    plt.title(title)
    plt.legend()
    plt.show()

forecast_and_evaluate(df, 'Forecast on Original Data')
forecast_and_evaluate(df_clean, 'Forecast on Cleaned Data')

## 5. Visualization for Stakeholders

This section provides a clear, non-technical visualization of the detected anomalies.

In [ ]:
plt.figure(figsize=(18, 8))
plt.plot(df.index, df['value'], label='Sales Data', color='black', alpha=0.3)
plt.scatter(anomalies_if.index, anomalies_if['value'], color='red', s=100, label='Unusual Spike/Dip (Point Anomaly)')
plt.scatter(anomalies_seasonal.index, anomalies_seasonal['value'], color='orange', s=100, label='Unusual for the Season (Contextual Anomaly)')
plt.axvline(x=df.index[break_point_to_test], color='purple', linestyle='--', label='Market Shift (Structural Break)')

plt.title('Anomaly Report for Sales Data')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.show()